# PPO em um Mini-GridWorld

> Parte da série [ML Notebooks](../README.md) — por **Nandobez**.


## Intuição

PPO é um algoritmo actor-critic que limita o quanto a política pode mudar em uma única atualização via uma razão de probabilidade recortada. Mantém o treino estável continuando on-policy e fácil de implementar.


## Formulação Matemática

$$r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)}$$

$$\mathcal{L}^{\text{CLIP}} = \mathbb{E}_t\!\left[\min\big(r_t \hat A_t,\ \text{clip}(r_t, 1-\varepsilon, 1+\varepsilon)\hat A_t\big)\right]$$


## Implementação


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np


In [ ]:
class GridWorld:
    """5x5 toy world; the agent must reach (4,4)."""
    def __init__(self, size=5):
        self.size = size
        self.reset()
    def reset(self):
        self.s = (0, 0)
        return self._obs()
    def _obs(self):
        v = np.zeros(self.size * self.size)
        v[self.s[0] * self.size + self.s[1]] = 1.0
        return v
    def step(self, a):
        x, y = self.s
        dx, dy = [(-1,0),(1,0),(0,-1),(0,1)][a]
        x = max(0, min(self.size - 1, x + dx))
        y = max(0, min(self.size - 1, y + dy))
        self.s = (x, y)
        done = self.s == (self.size - 1, self.size - 1)
        return self._obs(), 1.0 if done else -0.01, done

class ActorCritic(nn.Module):
    def __init__(self, obs_dim, n_actions):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(obs_dim, 64), nn.ReLU())
        self.pi = nn.Linear(64, n_actions)
        self.v = nn.Linear(64, 1)
    def forward(self, x):
        h = self.shared(x)
        return self.pi(h), self.v(h).squeeze(-1)


## Experimento


In [ ]:
env = GridWorld()
obs_dim = env.size * env.size
model = ActorCritic(obs_dim, 4)
opt = torch.optim.Adam(model.parameters(), lr=3e-3)
gamma, lam, eps = 0.99, 0.95, 0.2

for iter_ in range(200):
    obs = env.reset()
    states, actions, log_probs, rewards, values, dones = [], [], [], [], [], []
    for _ in range(64):
        s = torch.tensor(obs, dtype=torch.float32)
        logits, v = model(s)
        dist = torch.distributions.Categorical(logits=logits)
        a = dist.sample()
        obs2, r, d = env.step(a.item())
        states.append(s); actions.append(a); log_probs.append(dist.log_prob(a))
        rewards.append(r); values.append(v); dones.append(d)
        obs = env.reset() if d else obs2
    values_t = torch.stack(values + [torch.tensor(0.0)])
    rewards_t = torch.tensor(rewards)
    dones_t = torch.tensor(dones, dtype=torch.float32)
    adv = torch.zeros_like(rewards_t)
    gae = 0
    for t in reversed(range(len(rewards))):
        delta = rewards_t[t] + gamma * values_t[t+1] * (1 - dones_t[t]) - values_t[t]
        gae = delta + gamma * lam * (1 - dones_t[t]) * gae
        adv[t] = gae
    returns = adv + values_t[:-1].detach()
    adv = (adv - adv.mean()) / (adv.std() + 1e-8)

    s_b = torch.stack(states); a_b = torch.stack(actions)
    old_logp = torch.stack(log_probs).detach()
    for _ in range(4):  # PPO epochs
        logits, v = model(s_b)
        dist = torch.distributions.Categorical(logits=logits)
        new_logp = dist.log_prob(a_b)
        ratio = (new_logp - old_logp).exp()
        l1 = ratio * adv
        l2 = ratio.clamp(1 - eps, 1 + eps) * adv
        policy_loss = -torch.min(l1, l2).mean()
        value_loss = F.mse_loss(v, returns)
        loss = policy_loss + 0.5 * value_loss - 0.01 * dist.entropy().mean()
        opt.zero_grad(); loss.backward(); opt.step()
    if iter_ % 25 == 0:
        print(f'iter {iter_:3d}  policy loss {policy_loss.item():.3f}  value loss {value_loss.item():.3f}')


## Discussão

- O threshold $\varepsilon = 0.2$ é o padrão recomendado.
- GAE (aqui $\lambda=0.95$) torna as estimativas de advantage bem mais suaves.
- Para ambientes reais adicione normalização de observação e rollouts paralelos.


## Referências

- Repositório da série: [github.com/Nandobez/ml-notebooks](https://github.com/Nandobez/ml-notebooks)
- Autor: [Nandobez](https://github.com/Nandobez)
